# LeakMind Evaluation & Benchmark Notebook

This notebook evaluates the production models against the CERT insider threat benchmark datasets.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

from src.intelligence import LeakPredictor, TrustScoreEngine, LeakExplainability
from src.decision import PolicyEngine

## 1. Load Test Dataset & Models

In [ ]:
X_test = pd.read_csv('../data/processed/X_daily_supervised.csv')
y_test = pd.read_csv('../data/processed/y_daily_supervised.csv').iloc[:, 0]

print(f'Test feature matrix shape: {X_test.shape}')
print('Target class distribution:\n', y_test.value_counts())

## 2. Supervised Benchmark (XGBoost & Random Forest)

In [ ]:
models = {
    'XGBoost': joblib.load('../models/xgboost_model.joblib'),
    'Random Forest': joblib.load('../models/rf_model.joblib')
}

for name, m in models.items():
    preds = m.predict(X_test)
    probs = m.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    print(f'=== {name} Performance ===')
    print(f'Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}')
    print(classification_report(y_test, preds, target_names=['Normal', 'Insider']))

## 3. End-to-End Pipeline & Policy Enforcement

In [ ]:
predictor = LeakPredictor(models_dir='../models')
policy = PolicyEngine()
explainer = LeakExplainability(models_dir='../models')

sample = X_test.iloc[0].to_dict()
pred_res = predictor.predict_one(sample)
action = policy.evaluate_risk(pred_res['risk_score'])
explanation = explainer.explain_sample(sample)

print('Prediction Result:', pred_res)
print('Enforced Policy Action:', action.value)
print('SHAP Explanation Narrative:', explanation['narrative'])